# VACUUM — Todas as Tabelas Delta Lake

Remove arquivos físicos de versões antigas de todas as tabelas Delta.

| Tabela | Path |
|--------|------|
| bronze | `s3a://bronze/pda/beneficios-emitidos/` |
| silver | `s3a://prata/pda/beneficios-emitidos/` |
| gold/fat_uf | `s3a://ouro/pda/beneficios-emitidos/fat_uf/` |
| gold/fat_especie | `s3a://ouro/pda/beneficios-emitidos/fat_especie/` |
| gold/fat_banco | `s3a://ouro/pda/beneficios-emitidos/fat_banco/` |
| gold/kpis_nacionais | `s3a://ouro/pda/beneficios-emitidos/kpis_nacionais/` |

> ⚠️ `RETAIN 0 HOURS` remove imediatamente todos os arquivos não referenciados pela versão atual. Use `RETAIN 168 HOURS` (7 dias) em produção com time travel ativo.

In [34]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("Vacuum-All-Delta-Tables")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Permite RETAIN 0 HOURS (desativa checagem de segurança)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

print(f"Spark {spark.version} | Delta pronto")

Spark 3.5.5 | Delta pronto


In [10]:
import time

TABELAS = [
    ("bronze",           "s3a://bronze/pda/beneficios-emitidos/"),
    ("silver",           "s3a://prata/pda/beneficios-emitidos/"),
    ("gold/fat_uf",      "s3a://ouro/pda/beneficios-emitidos/fat_uf/"),
    ("gold/fat_especie", "s3a://ouro/pda/beneficios-emitidos/fat_especie/"),
    ("gold/fat_banco",   "s3a://ouro/pda/beneficios-emitidos/fat_banco/"),
    ("gold/kpis",        "s3a://ouro/pda/beneficios-emitidos/kpis_nacionais/"),
]

print(f"{'Tabela':<22} {'Status':<12} {'Tempo':>8}")
print("-" * 45)

Tabela                 Status          Tempo
---------------------------------------------


In [41]:


for nome, path in TABELAS:
    t0 = time.time()
    try:
        spark.sql(f"VACUUM delta.`{path}` RETAIN 0 HOURS")
        elapsed = time.time() - t0
        print(f"{nome:<22} {'OK':<12} {elapsed:>7.1f}s")
    except Exception as e:
        elapsed = time.time() - t0
        print(f"{nome:<22} {'ERRO':<12} {elapsed:>7.1f}s  →  {e}")

print("-" * 45)
print("VACUUM concluído em todas as tabelas.")

Deleted 2 files and directories in a total of 2 directories.
bronze                 OK              26.4s


Deleted 0 files and directories in a total of 2 directories.
silver                 OK              20.9s


Deleted 1 files and directories in a total of 2 directories.
gold/fat_uf            OK              18.7s


Deleted 5 files and directories in a total of 2 directories.
gold/fat_especie       OK              18.2s


Deleted 1 files and directories in a total of 2 directories.
gold/fat_banco         OK              17.7s


Deleted 1 files and directories in a total of 2 directories.
gold/kpis              OK              17.7s
---------------------------------------------
VACUUM concluído em todas as tabelas.


In [42]:
# Histórico Delta de cada tabela após o VACUUM
for nome, path in TABELAS:
    print(f"\n── {nome} ──────────────────────")
    spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select(
        "version", "timestamp", "operation", "operationParameters","operationMetrics"
    ).show(3, truncate=60)


── bronze ──────────────────────
+-------+-------------------+---------+------------------------------------------------------------+------------------------------------------------------------+
|version|          timestamp|operation|                                         operationParameters|                                            operationMetrics|
+-------+-------------------+---------+------------------------------------------------------------+------------------------------------------------------------+
|      2|2026-03-04 01:35:11|    WRITE|{mode -> Overwrite, partitionBy -> ["_ano_mes"], predicat...|{numFiles -> 2, numRemovedFiles -> 2, numRemovedBytes -> ...|
|      1|2026-03-04 00:30:05|    WRITE|{mode -> Overwrite, partitionBy -> ["_ano_mes"], predicat...|{numFiles -> 2, numRemovedFiles -> 3, numRemovedBytes -> ...|
|      0|2026-03-03 03:33:51|    WRITE|{mode -> Overwrite, partitionBy -> ["_ano_mes"], predicat...|{numFiles -> 3, numCopiedRows -> 0, numAddedChangeFiles 

In [13]:
hist = spark.sql(f"DESCRIBE HISTORY delta.`s3a://bronze/pda/beneficios-emitidos/`")
print(hist.columns)
hist.printSchema()

['version', 'timestamp', 'userId', 'userName', 'operation', 'operationParameters', 'job', 'notebook', 'clusterId', 'readVersion', 'isolationLevel', 'isBlindAppend', 'operationMetrics', 'userMetadata', 'engineInfo']
root
 |-- version: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- userId: string (nullable = true)
 |-- userName: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- operationParameters: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- job: struct (nullable = true)
 |    |-- jobId: string (nullable = true)
 |    |-- jobName: string (nullable = true)
 |    |-- jobRunId: string (nullable = true)
 |    |-- runId: string (nullable = true)
 |    |-- jobOwnerId: string (nullable = true)
 |    |-- triggerType: string (nullable = true)
 |-- notebook: struct (nullable = true)
 |    |-- notebookId: string (nullable = true)
 |-- clusterId: string (nullable = true)
 |-- readVersion: long (null

In [20]:
from pyspark.sql import functions as F

hist = spark.sql(f"DESCRIBE HISTORY delta.`s3a://ouro/pda/beneficios-emitidos/fat_banco/`")

(hist
 .filter(F.col("operation") == "MERGE")
 .select(
     "version",
     "timestamp",
     F.element_at("operationMetrics", F.lit("numTargetRowsUpdated")).alias("rows_updated"),
     F.element_at("operationMetrics", F.lit("numTargetRowsInserted")).alias("rows_inserted"),
     F.element_at("operationMetrics", F.lit("numTargetRowsDeleted")).alias("rows_deleted"),
     F.element_at("operationMetrics", F.lit("numSourceRows")).alias("source_rows"),
     "operationParameters"
 )
 .orderBy(F.col("version").desc())
 .show(20, truncate=False)
)

+-------+-------------------+------------+-------------+------------+-----------+----------------------------------------+
|version|timestamp          |rows_updated|rows_inserted|rows_deleted|source_rows|operationParameters                     |
+-------+-------------------+------------+-------------+------------+-----------+----------------------------------------+
|2      |2026-03-04 01:08:36|NULL        |NULL         |NULL        |NULL       |{queryId -> 20260304_010834_00036_g36wg}|
+-------+-------------------+------------+-------------+------------+-----------+----------------------------------------+



AnalysisException: [REQUIRES_SINGLE_PART_NAMESPACE] spark_catalog requires a single-part namespace, but got `delta`.`ouro`.

In [21]:
# Ler a versão ANTES do MERGE (versão 1)
df_before = spark.read.format("delta").option("versionAsOf", 1).table("delta.ouro.fat_banco")

# Ler a versão DEPOIS do MERGE (versão 2)
df_after = spark.read.format("delta").option("versionAsOf", 2).table("delta.ouro.fat_banco")

# Comparar linhas onde banco_codigo = 237
df_before.filter("banco_codigo = 237").show()
df_after.filter("banco_codigo = 237").show()

AnalysisException: [REQUIRES_SINGLE_PART_NAMESPACE] spark_catalog requires a single-part namespace, but got `delta`.`ouro`.

In [26]:
# Em vez de delta.ouro.fat_banco
spark.sql("DESCRIBE HISTORY delta.ouro.fat_banco").show(truncate=False)

AnalysisException: [REQUIRES_SINGLE_PART_NAMESPACE] spark_catalog requires a single-part namespace, but got `delta`.`ouro`.

In [28]:
path = "s3a://ouro/pda/beneficios-emitidos/fat_banco/"  # ajuste
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").show(truncate=False)

+-------+-------------------+------+--------+---------+----------------------------------------------------------------------------------+----+--------+---------------------------+-----------+-----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                                                               |job |notebook|clusterId                  |readVersion|isolationLevel   |isBlindAppend|operationMetrics                                                                                                                                                               |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+---------------------------------------------

In [35]:
delta_table = DeltaTable.forPath(spark,'s3a://ouro/pda/beneficios-emitidos/fat_banco/')

In [36]:
history_df = delta_table.history()

In [40]:
history_df.select("version","timestamp","operation","operationParameters","operationMetrics").show(truncate=False)

+-------+-------------------+---------+----------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                               |operationMetrics                                                                                                                                                               |
+-------+-------------------+---------+----------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|4      |2026-03-04 01:51:35|WRITE    |{mode -> Overwrite, partitionBy -> ["_ano_mes"], predicate -> _a

In [33]:
df_after.count()

21

In [5]:
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
